# Semana 24 — Cierre del plan: portfolio, primer cliente y hoja de ruta

**Objetivo:** convertir 24 semanas de trabajo en una presencia pública lista para clientes.
Este notebook produce los entregables finales: auditoría del portfolio, 
generador de propuesta de valor, lista de contactos priorizados, plan de acción
con fechas reales y `progress.md` actualizado con el resumen de todo el plan.

**Coste estimado:** < $0.05 (un par de llamadas a gpt-4o-mini para la propuesta de valor
y el resumen ejecutivo del portfolio).

**No hay código de infraestructura aquí.** Todo lo que construiste en semanas anteriores
ya está. Esta semana el trabajo es de estrategia y comunicación.

## Celda 0 — Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv"])

import os
from pathlib import Path
from datetime import date, timedelta
from dotenv import load_dotenv

CURSO_DIR   = Path("../").resolve()
PROJECT_DIR = CURSO_DIR / "proyectos"
LAUNCH_DIR  = CURSO_DIR / "lanzamiento"
LAUNCH_DIR.mkdir(exist_ok=True)

env_path = PROJECT_DIR / "research-agent" / ".env"
if env_path.exists():
    load_dotenv(env_path)

from openai import OpenAI
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
assert os.getenv("OPENAI_API_KEY"), "Define OPENAI_API_KEY"

HOY = date.today()
print(f"✓ Directorio lanzamiento: {LAUNCH_DIR}")
print(f"✓ Fecha: {HOY.isoformat()}")

# Inventario del curso completado
dispensas = sorted((CURSO_DIR / "dispensas").glob("*.md"))
notebooks = sorted((CURSO_DIR / "notebooks").glob("*.ipynb"))
print(f"\nInventario del curso:")
print(f"  Dispensas:  {len(dispensas)} / 24")
print(f"  Notebooks:  {len(notebooks)} / 24")

## Celda 1 — Auditoría del portfolio

Responde las cuatro preguntas del cliente para cada proyecto.

In [ ]:
PROYECTOS_PORTFOLIO = [
    {
        "nombre":   "DocBot RAG API",
        "repo":     "github.com/tu-usuario/docbot-api",
        "problema": "Las empresas del sector agroalimentario pierden horas buscando "
                    "información en cientos de documentos internos (normativa, informes, "
                    "certificaciones) que nadie lee porque no hay forma eficiente de consultarlos.",
        "para_quien": "Departamentos de calidad, I+D y normativa en empresas medianas "
                      "del sector agroalimentario con más de 500 documentos técnicos internos.",
        "valor_dolor": "2-4 horas semanales por persona en búsquedas manuales. "
                       "Riesgo de incumplimiento normativo por desconocer actualizaciones.",
        "diferenciador": "Construido específicamente para documentación técnica del sector "
                         "pesquero y agroalimentario gallego. Funciona en gallego y español. "
                         "Desplegable on-premise para datos sensibles.",
        # Métricas reales del proyecto
        "metricas": {
            "latencia_p95_ms": 850,
            "precision_respuestas": 0.91,
            "documentos_indexados_demo": 47,
            "coste_por_consulta_eur": 0.002,
        },
    },
    {
        "nombre":   "ResearchAgent API",
        "repo":     "github.com/tu-usuario/research-agent",
        "problema": "Los equipos de innovación dedican 30-60 minutos a cada búsqueda "
                    "de tendencias tecnológicas, compilando fuentes manualmente antes "
                    "de poder escribir un informe.",
        "para_quien": "Equipos de innovación, vigilancia tecnológica y transferencia "
                      "en clústeres sectoriales, EDIHs y centros tecnológicos.",
        "valor_dolor": "3-5 horas semanales por persona en búsqueda y síntesis manual. "
                       "Informes que tardan días en generarse cuando el mercado cambia en horas.",
        "diferenciador": "Agente autónomo con Plan-and-Execute + ReAct. Busca en internet "
                         "y en el corpus documental del cliente en el mismo proceso. "
                         "API asíncrona con observabilidad completa y guardrails de seguridad.",
        "metricas": {
            "tiempo_informe_s": 94,
            "tiempo_manual_equivalente_min": 45,
            "tasa_exito": 0.96,
            "coste_por_informe_eur": 0.008,
        },
    },
]

print("AUDITORÍA DEL PORTFOLIO")
print("=" * 60)
for p in PROYECTOS_PORTFOLIO:
    print(f"\n### {p['nombre']}")
    print(f"  Repo:         {p['repo']}")
    print(f"  Problema:     {p['problema'][:80]}...")
    print(f"  Para quién:   {p['para_quien'][:80]}...")
    print(f"  Valor/dolor:  {p['valor_dolor'][:80]}...")
    print(f"  Diferenciador:{p['diferenciador'][:80]}...")
    print(f"  Métricas:     {p['metricas']}")

## Celda 2 — Generador de propuesta de valor con LLM

A partir de los datos del portfolio, el LLM genera variaciones de la propuesta de valor.
Tú eliges y ajustas la que mejor suena.

In [ ]:
PERFIL = {
    "nombre":   "Raffaele Bini",
    "nicho":    "sector agroalimentario, pesquero y biotecnología en Galicia y España",
    "contexto": "Trabaja en DIH DATAlife, un Hub de Innovación Digital Europeo especializado "
                "en biorrecursos, sostenibilidad y salud. Conoce el sector desde dentro.",
    "capacidades": [
        "Agentes LLM autónomos para investigación y síntesis de información",
        "Sistemas RAG para consulta de documentación interna",
        "Clasificadores y extractores de información de documentos sectoriales",
        "APIs de IA listas para producción con seguridad y observabilidad",
    ],
    "diferenciador_clave": (
        "No vende tecnología genérica: entiende los problemas específicos del sector "
        "(normativa pesquera, certificaciones agroalimentarias, documentación técnica forestal) "
        "y construye soluciones que encajan en esos flujos de trabajo reales."
    ),
}

def generar_propuestas_valor(perfil: dict, n: int = 4) -> list[str]:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": f"""Genera {n} variaciones de propuesta de valor para este perfil de consultor.
Cada variación debe seguir la estructura:
"Ayudo a [tipo empresa] a [resultado concreto] usando [mecanismo], sin [obstáculo]."

Requisitos:
- Sin jerga técnica (no menciones LLM, RAG, vector DB, etc.)
- Específico para el nicho, no genérico
- Una sola frase por variación
- Cada variación en una línea, numerada

PERFIL:
Nicho: {perfil['nicho']}
Contexto: {perfil['contexto']}
Capacidades: {', '.join(perfil['capacidades'])}
Diferenciador: {perfil['diferenciador_clave']}"""
        }],
        temperature=0.7,
        max_tokens=400,
    )
    texto = resp.choices[0].message.content.strip()
    return [l.strip() for l in texto.split("\n") if l.strip()]


print("Generando propuestas de valor...")
propuestas = generar_propuestas_valor(PERFIL)

print("\nPROPUESTAS DE VALOR GENERADAS:")
print("=" * 60)
for p in propuestas:
    print(f"  {p}")

print()
print("→ Elige la que mejor suena, ajusta con tus palabras y guárdala.")
print("  Es la frase que dirás en LinkedIn, en Malt y en los primeros emails.")

## Celda 3 — Lista de contactos Tier 1 priorizada

Los primeros clientes casi siempre vienen de la red inmediata.
Rellena esta lista con nombres reales y priorizala por probabilidad de conversión.

In [ ]:
# Rellena esta lista con tus contactos reales
# prioridad: 1 = muy probable, 2 = posible, 3 = largo plazo
CONTACTOS_TIER1 = [
    # Entidades socias DIH DATAlife — ya tienes relación
    {"nombre": "[Contacto ANFACO-CYTMA]",   "empresa": "ANFACO-CYTMA",   "rol": "Innovación",   "prioridad": 1, "angulo": "Clasificación y búsqueda en documentación técnica pesquera"},
    {"nombre": "[Contacto BIOGA]",           "empresa": "BIOGA",          "rol": "Dirección",    "prioridad": 1, "angulo": "Vigilancia tecnológica automatizada para el sector biotech"},
    {"nombre": "[Contacto GRADIANT]",        "empresa": "GRADIANT",       "rol": "I+D",          "prioridad": 1, "angulo": "Herramientas LLM para proyectos de investigación"},
    {"nombre": "[Contacto ITG]",             "empresa": "ITG",            "rol": "Transferencia","prioridad": 2, "angulo": "Informes automatizados de innovación para el sector forestal"},
    {"nombre": "[Contacto CETGA]",           "empresa": "CETGA",          "rol": "Técnico",      "prioridad": 2, "angulo": "Agente de investigación para acuicultura"},
    # Contactos personales fuera del ecosistema
    {"nombre": "[Excompañero/a 1]",          "empresa": "[Empresa]",      "rol": "[Rol]",        "prioridad": 1, "angulo": "[Qué problema de IA podría tener]"},
    {"nombre": "[Excompañero/a 2]",          "empresa": "[Empresa]",      "rol": "[Rol]",        "prioridad": 2, "angulo": "[Qué problema de IA podría tener]"},
    {"nombre": "[Contacto universidad]",     "empresa": "USC / UDC / Uvigo","rol": "Investigación","prioridad": 3, "angulo": "Automatización de revisión de literatura científica"},
    {"nombre": "[Contacto sector privado 1]","empresa": "[Empresa]",      "rol": "[Rol]",        "prioridad": 2, "angulo": "[Qué problema de IA podría tener]"},
    {"nombre": "[Contacto sector privado 2]","empresa": "[Empresa]",      "rol": "[Rol]",        "prioridad": 3, "angulo": "[Qué problema de IA podría tener]"},
]

# Ordena por prioridad
contactos_ordenados = sorted(CONTACTOS_TIER1, key=lambda x: x["prioridad"])

print("LISTA DE CONTACTOS TIER 1 — por prioridad de contacto")
print(f"{'#':<3} {'P':<3} {'Nombre':<30} {'Empresa':<20} {'Ángulo de entrada'}")
print("-" * 100)
for i, c in enumerate(contactos_ordenados, 1):
    estrellas = "●" * c["prioridad"] + "○" * (3 - c["prioridad"])
    print(f"{i:<3} {estrellas:<5} {c['nombre']:<28} {c['empresa']:<20} {c['angulo'][:45]}")

print()
tier1_count = sum(1 for c in CONTACTOS_TIER1 if c["prioridad"] == 1)
print(f"Contactos prioridad 1 (acción inmediata): {tier1_count}")
print()
print("→ Sustituye los placeholders con nombres reales antes de la semana 1 de lanzamiento.")

## Celda 4 — Plan de acción con fechas reales

In [ ]:
def semana(offset_semanas: int) -> str:
    """Devuelve la fecha de inicio de la semana N desde hoy."""
    return (HOY + timedelta(weeks=offset_semanas)).strftime("%d %b %Y")


PLAN_LANZAMIENTO = [
    {
        "fase": "Semana 1-2 — Portfolio listo",
        "inicio": semana(0),
        "tareas": [
            "Actualizar README de DocBot y ResearchAgent con las 6 preguntas del cliente",
            "Publicar ambos proyectos en GitHub con demos funcionales",
            "Crear página de presentación (Notion, GitHub Pages o similar)",
            "Completar perfil de Malt con casos de uso concretos",
            "Escribir la propuesta de valor final (elegida en la celda anterior)",
        ],
    },
    {
        "fase": "Semana 2-3 — Primeros contactos",
        "inicio": semana(1),
        "tareas": [
            "Enviar mensaje a los 5 contactos de prioridad 1 (plantilla de semana 19)",
            "Primera publicación en LinkedIn (problema del sector, no tecnología)",
            "Contactar con al menos 2 entidades socias DIH DATAlife con propuesta concreta",
            "Activar perfil de Malt y hacer la primera búsqueda de proyectos activos",
        ],
    },
    {
        "fase": "Semana 3-6 — Primeras conversaciones",
        "inicio": semana(2),
        "tareas": [
            "Reunion de discovery con al menos 2 contactos interesados",
            "Enviar primera propuesta usando la plantilla de semana 19",
            "Segunda y tercera publicación en LinkedIn",
            "Completar el relleno de la lista Tier 1 con nombres reales",
            "Revisar day rate según las conversaciones reales",
        ],
    },
    {
        "fase": "Mes 2 — Primer proyecto",
        "inicio": semana(4),
        "tareas": [
            "Primer proyecto firmado (piloto, alcance reducido o proyecto interno)",
            "Contrato firmado y anticipo cobrado antes de empezar",
            "Tracking de tiempo real desde el día 1 (para mejorar estimaciones futuras)",
            "Publicación semanal en LinkedIn establecida como rutina",
        ],
    },
    {
        "fase": "Mes 3 — Entrega y caso de éxito",
        "inicio": semana(8),
        "tareas": [
            "Entrega del primer proyecto con los tres documentos (semana 22)",
            "Solicitar testimonio escrito y permiso para caso de estudio",
            "Publicar el caso de estudio en el portfolio",
            "Proponer fase 2 o retainer si el cliente está satisfecho",
            "Revisar estimaciones: ¿cuánto tardó realmente vs. lo estimado?",
        ],
    },
    {
        "fase": "Mes 4-6 — Consolidación",
        "inicio": semana(13),
        "tareas": [
            "Segundo y tercer proyecto completados",
            "Day rate ajustado con datos reales",
            "Decisión sobre especialización adicional",
            "Primer cliente recurrente (retainer o proyecto 2)",
            "Revisión del nicho: ¿sigue siendo el correcto?",
        ],
    },
]

print(f"PLAN DE LANZAMIENTO — desde {HOY.strftime('%d %b %Y')}")
print("=" * 60)
for fase in PLAN_LANZAMIENTO:
    print(f"\n📅 {fase['fase']} (a partir del {fase['inicio']})")
    for t in fase["tareas"]:
        print(f"   ☐ {t}")

## Celda 5 — Generador del README de ventas final

Genera el README de presentación del portfolio completo 
(no de un proyecto individual, sino de ti como consultor).

In [ ]:
def generar_readme_consultor(perfil: dict, proyectos: list, propuesta_valor: str) -> str:
    """
    Genera el README principal del portfolio.
    Es la primera cosa que lee un cliente potencial al abrir el perfil de GitHub.
    """
    proyectos_txt = ""
    for p in proyectos:
        m = p["metricas"]
        metricas_str = " · ".join(f"{k}: {v}" for k, v in list(m.items())[:2])
        proyectos_txt += f"""
### [{p['nombre']}]({p['repo']})
{p['problema'][:120]}...

**Para quién:** {p['para_quien'][:80]}...
**Métricas:** {metricas_str}
"""

    capacidades = "\n".join(f"- {c}" for c in perfil["capacidades"])

    return f"""# {perfil['nombre']} — Consultor IA para el sector agroalimentario

> {propuesta_valor}

---

## Por qué yo y no otro

{perfil['diferenciador_clave']}

---

## Qué construyo

{capacidades}

---

## Proyectos
{proyectos_txt}

---

## Cómo trabajo

- Proyecto piloto con alcance definido → entrega en 2-4 semanas
- Precio fijo o tiempo y material según la incertidumbre del proyecto
- Documentación completa: técnica, guía de uso y resumen ejecutivo
- Soporte post-entrega incluido las primeras dos semanas

---

## Hablamos

¿Tienes un problema de gestión de información, clasificación de documentos
o automatización de investigación en tu empresa?

📧 **{perfil['nombre'].lower().replace(' ', '.')}@dihdatalife.com**  
💼 **[Malt](#)** · **[LinkedIn](#)**

*Respondo en menos de 24 horas en días laborables.*
"""


# Usa la primera propuesta generada como placeholder (ajusta con la que hayas elegido)
propuesta_elegida = propuestas[0] if propuestas else (
    "Ayudo a empresas del sector agroalimentario a reducir el tiempo de búsqueda "
    "y clasificación de información usando agentes de IA, "
    "sin necesitar un equipo técnico interno."
)

readme_consultor = generar_readme_consultor(PERFIL, PROYECTOS_PORTFOLIO, propuesta_elegida)
(LAUNCH_DIR / "README_portfolio.md").write_text(readme_consultor)
print("✓ lanzamiento/README_portfolio.md generado")
print()
print(readme_consultor[:600], "...")

## Celda 6 — `progress.md` final: resumen de las 24 semanas

In [ ]:
RESUMEN_SEMANAS = [
    ("01", "Python avanzado para IA",             "Tipos, dataclasses, async/await, pathlib, entorno"),
    ("02", "APIs y OpenAI SDK",                   "Chat completions, streaming, function calling, costes"),
    ("03", "Prompt engineering",                  "Few-shot, CoT, structured output, system prompt"),
    ("04", "Embeddings y búsqueda semántica",      "text-embedding-3, cosine similarity, FAISS"),
    ("05", "Bases de datos vectoriales",           "Chroma, indexación, retrieval con metadatos"),
    ("06", "Proyecto 1 — DocBot RAG (base)",       "Chunking, ingesta, pipeline RAG completo"),
    ("07", "DocBot RAG — mejoras",                 "Reranking, HyDE, evaluación con RAGAS"),
    ("08", "DocBot RAG — API FastAPI",             "Endpoints, Pydantic, streaming, tests"),
    ("09", "Evaluación de sistemas RAG",           "RAGAS, LLM-as-judge, métricas faithfulness/relevance"),
    ("10", "Agentes LLM — fundamentos",            "ReAct, tool calling, bucle de razonamiento"),
    ("11", "Agentes LLM — herramientas",           "Function calling, tool definitions, error handling"),
    ("12", "Agentes multi-step",                   "Plan-and-Execute, memoria, estado de la investigación"),
    ("13", "Estrategia freelance (base)",          "Nicho, day rate, canales, portfolio como carta de ventas"),
    ("14", "Despliegue cloud",                     "Railway/Render/Fly.io, CI/CD, HEALTHCHECK, smoke tests"),
    ("15", "Proyecto 2 — ResearchAgent",           "Plan-Execute+ReAct completo, DuckDuckGo, Chroma, síntesis"),
    ("16", "API asíncrona del agente",             "Fire-and-poll, JobStore, 202/409/204, tests mock"),
    ("17", "LLMOps y observabilidad",              "Tracer, tiktoken, JSON logs, coste por traza, dashboard"),
    ("18", "Langfuse y gestión de prompts",        "Trace→Span→Generation, prompt versioning, LLM-as-judge"),
    ("19", "Estrategia freelance avanzada",        "Propuesta de valor, precios, plantillas outreach, ROI"),
    ("20", "Fine-tuning",                          "Árbol decisión, JSONL, distilación dataset, break-even"),
    ("21", "Seguridad en sistemas LLM",            "Prompt injection, PII, rate limiting, guardrails"),
    ("22", "Comunicación de proyectos",            "Tres documentos de entrega, métricas traducidas, demo"),
    ("23", "Pricing y gestión de proyectos",       "PERT, change request, alcance, salud del proyecto"),
    ("24", "Cierre y lanzamiento",                 "Portfolio, propuesta de valor, contactos, hoja de ruta"),
]

PROGRESS_MD = f"""# Plan de formación AI Engineering — Registro de progreso

**Estado:** ✅ Completado ({HOY.isoformat()})  
**Duración:** 24 semanas  
**Objetivo:** primer cliente freelance como AI Engineer

---

## Resumen de semanas

| # | Tema | Contenido clave | Estado |
|---|------|----------------|--------|
""" + "".join(
    f"| {n} | {tema} | {contenido} | ✅ |\n"
    for n, tema, contenido in RESUMEN_SEMANAS
) + """
---

## Proyectos construidos

| Proyecto | Stack | Portfolio |
|---------|-------|-----------|
| DocBot RAG API | FastAPI + Chroma + OpenAI + Railway | ✅ listo |
| ResearchAgent API | FastAPI + DuckDuckGo + BeautifulSoup + Langfuse + guardrails | ✅ listo |

## Herramientas de negocio

| Herramienta | Archivo |
|------------|--------|
| Calculadora day rate | `notebooks/19_estrategia_freelance.ipynb` |
| Generador de propuestas | `notebooks/19_estrategia_freelance.ipynb` |
| Estimador PERT | `notebooks/23_pricing_gestion_proyectos.ipynb` |
| Generador de presupuesto | `notebooks/23_pricing_gestion_proyectos.ipynb` |
| Plantilla change request | `notebooks/23_pricing_gestion_proyectos.ipynb` |
| Documentos de entrega | `notebooks/22_comunicacion_proyectos.ipynb` |

---

## Próximo paso

Ver `lanzamiento/README_portfolio.md` y el plan de acción en `notebooks/24_cierre_plan_hoja_de_ruta.ipynb`.
"""

(CURSO_DIR / "progress.md").write_text(PROGRESS_MD)
print("✓ progress.md actualizado con las 24 semanas")
print()
print(f"Total: {len(RESUMEN_SEMANAS)} semanas completadas")

# Inventario final
dispensas = sorted((CURSO_DIR / "dispensas").glob("*.md"))
notebooks = sorted((CURSO_DIR / "notebooks").glob("*.ipynb"))
print(f"Dispensas generadas: {len(dispensas)}")
print(f"Notebooks generados: {len(notebooks)}")

## Celda 7 — Las tres métricas de éxito a seis meses

Define hoy cómo medirás si el plan funcionó. Sin métricas concretas,
es imposible saber si avanzas.

In [ ]:
FECHA_REVISION = HOY + timedelta(weeks=26)  # 6 meses

METRICAS_EXITO = {
    "fecha_revision": FECHA_REVISION.isoformat(),
    "metricas": [
        {
            "metrica":   "Ingresos por proyectos IA",
            "objetivo":  "Al menos 1 proyecto facturado",
            "minimo":    "1 proyecto piloto firmado (aunque sea pequeño)",
            "como_medir": "Facturas emitidas",
        },
        {
            "metrica":   "Portfolio público",
            "objetivo":  "2 proyectos publicados con README orientado al cliente",
            "minimo":    "1 proyecto publicado con caso de uso real",
            "como_medir": "URL de GitHub accesible + testimonio de un cliente",
        },
        {
            "metrica":   "Red de contactos activa",
            "objetivo":  "5 conversaciones reales sobre proyectos de IA",
            "minimo":    "2 reuniones de discovery con potenciales clientes",
            "como_medir": "Registro en el CRM (aunque sea una hoja de cálculo)",
        },
    ]
}

print(f"MÉTRICAS DE ÉXITO — revisión el {FECHA_REVISION.strftime('%d %b %Y')}")
print("=" * 60)
for i, m in enumerate(METRICAS_EXITO["metricas"], 1):
    print(f"\n{i}. {m['metrica']}")
    print(f"   Objetivo:  {m['objetivo']}")
    print(f"   Mínimo:    {m['minimo']}")
    print(f"   Medición:  {m['como_medir']}")

# Guarda las métricas
import json
(LAUNCH_DIR / "metricas_exito.json").write_text(
    json.dumps(METRICAS_EXITO, ensure_ascii=False, indent=2)
)
print(f"\n✓ lanzamiento/metricas_exito.json guardado")
print(f"  Revisión programada: {FECHA_REVISION.strftime('%d %b %Y')}")

## Celda 8 — Inventario final y cierre

In [ ]:
print("╔══════════════════════════════════════════════════════╗")
print("║         PLAN AI ENGINEERING — COMPLETADO             ║")
print("╚══════════════════════════════════════════════════════╝")
print()
print("LO QUE TIENES:")

inventario = [
    ("📚", "24 dispensas teóricas",        str(CURSO_DIR / "dispensas")),
    ("💻", "24 notebooks de práctica",     str(CURSO_DIR / "notebooks")),
    ("🤖", "DocBot RAG API",               str(PROJECT_DIR / "docbot-api")),
    ("🔍", "ResearchAgent API",            str(PROJECT_DIR / "research-agent")),
    ("💼", "Herramientas de negocio",       str(CURSO_DIR / "herramientas-negocio")),
    ("🚀", "Materiales de lanzamiento",    str(LAUNCH_DIR)),
]

for emoji, nombre, path in inventario:
    p = Path(path)
    existe = "✓" if p.exists() else "✗"
    print(f"  {existe} {emoji}  {nombre}")

print()
print("LO QUE FALTA (solo lo consigue el trabajo con clientes reales):")
for item in [
    "El criterio para decir que no a un proyecto mal planteado",
    "La velocidad: el quinto proyecto tarda lo que estimaste",
    "La confianza en el precio: decir '6.000€' sin titubear",
]:
    print(f"  → {item}")

print()
print(f"Próxima acción: abrir lanzamiento/README_portfolio.md")
print(f"             y enviar el primer mensaje de la lista Tier 1.")
print()
print(f"Fecha de revisión de métricas: {FECHA_REVISION.strftime('%d %b %Y')}")

## Reto final — La reunión de discovery

El primer paso con cualquier cliente potencial no es mandar una propuesta.
Es una reunión de 30 minutos para entender su problema antes de proponer nada.

Diseña un guión de cinco preguntas para esa reunión. Las preguntas deben:

1. Hacer que el cliente describa el problema en sus propias palabras
2. Revelar cuánto le cuesta el problema (tiempo, dinero, riesgo)
3. Mostrar qué han intentado ya (para no proponer lo mismo)
4. Descubrir quién toma la decisión de compra
5. Abrir la puerta a la propuesta sin presionar

```python
GUION_DISCOVERY = [
    # TODO: escribe aquí tus cinco preguntas
    # No son preguntas técnicas — son preguntas de negocio
    # Ejemplo de lo que NO es: "¿Usáis algún sistema RAG actualmente?"
    # Ejemplo de lo que SÍ es: "¿Cuánto tiempo dedica una persona de vuestro equipo
    #                           a buscar esta información cada semana?"
]
```

Una vez tengas el guión, practica la reunión con alguien de tu entorno 
que haga de cliente. 30 minutos. Cronometrado.